# XGBoost + GLCM — 5-Fold Patient-Level CV (BraTS 2021)
## Protocol aligned for reporting alongside Saeed et al. (2023)

**Current study**
- Input: FLAIR-derived voxel-based GLCM radiomics.
- Target: Tumor vs Non-tumor **at voxel level**.
- Split unit: **patient_id** (no patient appears in both train and validation of a fold).
- Development set = original Train + Validation; Locked Test is **not loaded** here.
- 5-fold GroupKFold; class weights computed from the training portion of each fold only.
- Report Fold 1–5 ROC-AUC, Mean AUC, Variance AUC, SD, PR-AUC and auxiliary metrics.

**Saeed et al. (2023)**
- EfficientNet-B1 + FLAIR.
- Target: MGMT methylation status **at patient level**.
- 5-fold CV reporting in Table 4: ROC-AUC = 0.6128 / 0.5810 / 0.6300 / 0.6555 / 0.5546; Mean = 0.6068; Variance = 0.0016.

> **Important:** the two studies do **not** solve the same prediction task and do **not** use the same prediction unit. Therefore this notebook aligns the **validation/reporting protocol**, but does not interpret a higher/lower AUC as evidence that one model outperforms the other.


In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import GroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, balanced_accuracy_score
)

RANDOM_STATE = 42
N_SPLITS = 5

DATASET_PATH = "dataset/1200p/glcm/kernel5/"
TRAIN_PATH = os.path.join(DATASET_PATH, "dataset.train.csv")
VAL_PATH = os.path.join(DATASET_PATH, "dataset.validation.csv")
LOCKED_TEST_PATH = os.path.join(DATASET_PATH, "dataset.test.csv")  # DO NOT LOAD HERE

OUTPUT_DIR = "output/1200p/glcm/kernel5/patient_5fold_cv_saeed_comparison"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Values reported for EfficientNet-B1 + FLAIR in Saeed et al. (2023), Table 4.
SAEED_AUC_FOLDS = np.array([0.6128, 0.5810, 0.6300, 0.6555, 0.5546], dtype=float)
SAEED_REPORTED_MEAN_AUC = 0.6068
SAEED_REPORTED_VARIANCE_AUC = 0.0016

print("Protocol: 5-fold patient-grouped CV on Development only")
print("Locked Test path is defined but will NOT be loaded:", LOCKED_TEST_PATH)


## 1. Load Development only

The original Train and Validation files are merged only to form the **development cohort** for 5-fold CV. The locked Test file is not read in this notebook.


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

development = pd.concat([
    train_df.assign(original_split="train"),
    val_df.assign(original_split="validation")
], ignore_index=True)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Development:", development.shape)
print("Locked Test: NOT LOADED")
print("Columns:", development.columns.tolist())


## 2. Validate patient metadata and leakage

This step checks that splitting is truly patient-wise. A patient from the original Train set must not also occur in the original Validation set.


In [ ]:
PATIENT_CANDIDATES = ["patient_id", "patient", "subject_id", "case_id"]
patient_col = next((c for c in PATIENT_CANDIDATES if c in development.columns), None)

if patient_col is None:
    raise ValueError(
        "CSV thiếu patient_id. Không được chia 5-fold ngẫu nhiên theo voxel; "
        "hãy sửa bước tạo dataset trước."
    )
if "label" not in development.columns:
    raise ValueError("CSV thiếu cột label.")

coord_sets = [("x", "y", "z"), ("i", "j", "k"), ("voxel_x", "voxel_y", "voxel_z")]
coord_cols = next((list(cs) for cs in coord_sets if all(c in development.columns for c in cs)), None)

train_patients = set(train_df[patient_col].astype(str).unique())
val_patients = set(val_df[patient_col].astype(str).unique())
overlap = train_patients & val_patients
if overlap:
    raise ValueError(f"Patient leakage Train/Validation: {len(overlap)} patients")

# Standardize patient ID type once for the rest of the notebook.
development[patient_col] = development[patient_col].astype(str)

print("patient_col:", patient_col)
print("coordinates:", coord_cols)
print("development patients:", development[patient_col].nunique())
print("voxel labels:")
print(development["label"].value_counts().sort_index())

patient_audit = (
    development.groupby(patient_col)["label"]
    .agg(n_voxels="size", n_classes="nunique", tumor_voxels=lambda s: int((s == 1).sum()))
    .reset_index()
)
patient_audit["non_tumor_voxels"] = patient_audit["n_voxels"] - patient_audit["tumor_voxels"]
patient_audit.to_csv(os.path.join(OUTPUT_DIR, "development_patient_audit.csv"), index=False)

print("Patients containing both voxel classes:", int((patient_audit["n_classes"] == 2).sum()), "/", len(patient_audit))
if (patient_audit["n_classes"] < 2).any():
    print("WARNING: some patients contain only one voxel class in the sampled/extracted dataset.")


## 3. Select GLCM features

Only numeric GLCM radiomics features are used. Metadata, labels, patient IDs and voxel coordinates are excluded from the feature matrix.


In [ ]:
META = {
    "label", "original_split", patient_col,
    "patient_id", "patient", "subject_id", "case_id",
    "x", "y", "z", "i", "j", "k", "voxel_x", "voxel_y", "voxel_z"
}

numeric = [
    c for c in development.columns
    if c not in META and pd.api.types.is_numeric_dtype(development[c])
]
glcm = [c for c in numeric if "glcm" in c.lower()]

if not glcm:
    raise ValueError(
        "Không tìm thấy cột GLCM. Để so sánh đúng cấu hình luận văn XGBoost + GLCM, "
        "không tự động thay bằng toàn bộ numeric features."
    )

feature_cols = glcm
print("Number of GLCM features:", len(feature_cols))
print(*feature_cols, sep="\n")


## 4. Locked XGBoost configuration

These are the hyperparameters of the **current radiomics study**, not hyperparameters from Saeed et al. Keep them fixed during the 5 folds. Any tuning must be completed without using the locked Test set.


In [ ]:
XGB_CONFIG = dict(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    min_child_weight=1,
    gamma=0.0,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

with open(os.path.join(OUTPUT_DIR, "xgb_config.json"), "w", encoding="utf-8") as f:
    json.dump(XGB_CONFIG, f, indent=2)

print(json.dumps(XGB_CONFIG, indent=2))


## 5. 5-fold CV grouped by patient

`GroupKFold` guarantees that all voxels belonging to one patient stay in the same fold. Class weights are recalculated **inside each fold using only that fold's training voxels**.


In [ ]:
X = development[feature_cols]
y = development["label"].astype(int)
groups = development[patient_col].astype(str)

gkf = GroupKFold(n_splits=N_SPLITS)
rows = []
oof_parts = []
fold_patient_rows = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups), 1):
    Xtr, Xva = X.iloc[tr_idx], X.iloc[va_idx]
    ytr, yva = y.iloc[tr_idx], y.iloc[va_idx]
    gtr, gva = groups.iloc[tr_idx], groups.iloc[va_idx]

    train_patient_ids = set(gtr.unique())
    validation_patient_ids = set(gva.unique())
    patient_overlap = train_patient_ids & validation_patient_ids
    if patient_overlap:
        raise RuntimeError(f"Fold {fold}: patient leakage = {len(patient_overlap)}")
    if ytr.nunique() != 2 or yva.nunique() != 2:
        raise RuntimeError(f"Fold {fold}: training or validation portion lacks one voxel class")

    # Fold-specific class weighting: training portion only.
    cls = np.array([0, 1])
    w = compute_class_weight(class_weight="balanced", classes=cls, y=ytr)
    cw = dict(zip(cls, w))
    sample_weight = ytr.map(cw).to_numpy()

    model = XGBClassifier(**XGB_CONFIG)
    model.fit(Xtr, ytr, sample_weight=sample_weight, verbose=False)

    prob = model.predict_proba(Xva)[:, 1]
    pred = (prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(yva, pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    row = dict(
        fold=fold,
        train_patients=gtr.nunique(),
        validation_patients=gva.nunique(),
        train_voxels=len(tr_idx),
        validation_voxels=len(va_idx),
        train_non_tumor=int((ytr == 0).sum()),
        train_tumor=int((ytr == 1).sum()),
        validation_non_tumor=int((yva == 0).sum()),
        validation_tumor=int((yva == 1).sum()),
        validation_tumor_prevalence=float((yva == 1).mean()),
        class_weight_0=float(cw[0]),
        class_weight_1=float(cw[1]),
        roc_auc=roc_auc_score(yva, prob),
        pr_auc=average_precision_score(yva, prob),
        accuracy_t05=accuracy_score(yva, pred),
        balanced_accuracy_t05=balanced_accuracy_score(yva, pred),
        precision_t05=precision_score(yva, pred, zero_division=0),
        recall_t05=recall_score(yva, pred, zero_division=0),
        specificity_t05=specificity,
        f1_t05=f1_score(yva, pred, zero_division=0),
        mcc_t05=matthews_corrcoef(yva, pred),
        tn_t05=int(tn), fp_t05=int(fp), fn_t05=int(fn), tp_t05=int(tp)
    )
    rows.append(row)

    # Save OOF voxel predictions.
    cols = [patient_col, "label"] + (coord_cols or [])
    part = development.iloc[va_idx][cols].copy()
    part["fold"] = fold
    part["prob_tumor"] = prob
    part["pred_t05"] = pred
    oof_parts.append(part)

    # Save explicit patient-to-fold assignment for reproducibility.
    for pid in sorted(validation_patient_ids):
        fold_patient_rows.append({patient_col: pid, "validation_fold": fold})

    print(
        f"Fold {fold}: train patients={gtr.nunique()}, val patients={gva.nunique()}, "
        f"ROC-AUC={row['roc_auc']:.4f}, PR-AUC={row['pr_auc']:.4f}"
    )

fold_assignment = pd.DataFrame(fold_patient_rows)
if fold_assignment[patient_col].duplicated().any():
    raise RuntimeError("A patient appears in validation in more than one fold.")
if fold_assignment[patient_col].nunique() != development[patient_col].nunique():
    raise RuntimeError("Not every development patient appears exactly once as validation across 5 folds.")

fold_assignment.to_csv(os.path.join(OUTPUT_DIR, "patient_fold_assignment.csv"), index=False)


## 6. Fold 1–5, Mean, Variance and SD

To reproduce the convention visible in Saeed et al. Table 4, the primary `Variance AUC` is calculated as **sample variance (`ddof=1`)** across the five fold AUC values. For the published EfficientNet-B1 + FLAIR values, this gives approximately **0.001584 → 0.0016**, matching the reported table after rounding. Population variance (`ddof=0`) is also retained as a secondary value for transparency.


In [ ]:
fold_results = pd.DataFrame(rows)
auc = fold_results["roc_auc"].to_numpy(dtype=float)
prauc = fold_results["pr_auc"].to_numpy(dtype=float)

summary = pd.DataFrame([{
    "study": "Current study",
    "model": "XGBoost + GLCM",
    "modality": "FLAIR",
    "target": "Tumor vs Non-tumor voxel",
    "prediction_unit": "voxel (patient-separated CV)",
    "n_development_patients": int(development[patient_col].nunique()),
    "n_folds": N_SPLITS,
    "roc_auc_mean": auc.mean(),
    "roc_auc_variance_sample_ddof1": np.var(auc, ddof=1),
    "roc_auc_variance_population_ddof0": np.var(auc, ddof=0),
    "roc_auc_sd_sample": np.std(auc, ddof=1),
    "roc_auc_sd_population": np.std(auc, ddof=0),
    "pr_auc_mean": prauc.mean(),
    "pr_auc_variance_sample_ddof1": np.var(prauc, ddof=1),
    "pr_auc_variance_population_ddof0": np.var(prauc, ddof=0),
    "pr_auc_sd_sample": np.std(prauc, ddof=1),
    "pr_auc_sd_population": np.std(prauc, ddof=0)
}])

fold_results.to_csv(os.path.join(OUTPUT_DIR, "xgb_glcm_patient_5fold_metrics.csv"), index=False)
summary.to_csv(os.path.join(OUTPUT_DIR, "xgb_glcm_patient_5fold_summary.csv"), index=False)

display(fold_results[[
    "fold", "train_patients", "validation_patients",
    "validation_tumor_prevalence", "roc_auc", "pr_auc",
    "accuracy_t05", "balanced_accuracy_t05", "precision_t05",
    "recall_t05", "specificity_t05", "f1_t05", "mcc_t05"
]])
display(summary)


## 7. Saeed-style reporting table — with scientific comparability labels

This table reproduces the **reporting structure** (Fold 1–5, Mean AUC, Variance AUC) used by Saeed et al. It intentionally includes `Target`, `Prediction unit`, and `Direct performance comparison` columns.

**Do not calculate an AUC improvement/difference between the two rows.** Saeed et al. predict MGMT status at the patient level, whereas the current study classifies Tumor/Non-tumor voxels.


In [ ]:
current_row = {
    "Study": "Current study",
    "Dataset / input": "BraTS 2021 / FLAIR radiomics",
    "Target": "Tumor vs Non-tumor",
    "Prediction unit": "Voxel; folds separated by patient_id",
    "Model": "XGBoost + GLCM",
    "Validation": "5-fold GroupKFold by patient_id",
    "Fold 1 AUC": auc[0],
    "Fold 2 AUC": auc[1],
    "Fold 3 AUC": auc[2],
    "Fold 4 AUC": auc[3],
    "Fold 5 AUC": auc[4],
    "Mean AUC": auc.mean(),
    "Variance AUC (sample, ddof=1)": np.var(auc, ddof=1),
    "Direct performance comparison": "No — different target and prediction unit"
}

saeed_row = {
    "Study": "Saeed et al. (2023)",
    "Dataset / input": "BraTS 2021 Task 1 cases used for MGMT experiment / FLAIR",
    "Target": "MGMT methylation status",
    "Prediction unit": "Patient",
    "Model": "EfficientNet-B1",
    "Validation": "5-fold CV (reported by authors; exact fold assignment not reproduced here)",
    "Fold 1 AUC": SAEED_AUC_FOLDS[0],
    "Fold 2 AUC": SAEED_AUC_FOLDS[1],
    "Fold 3 AUC": SAEED_AUC_FOLDS[2],
    "Fold 4 AUC": SAEED_AUC_FOLDS[3],
    "Fold 5 AUC": SAEED_AUC_FOLDS[4],
    "Mean AUC": SAEED_REPORTED_MEAN_AUC,
    "Variance AUC (sample, ddof=1)": SAEED_REPORTED_VARIANCE_AUC,
    "Direct performance comparison": "No — different target and prediction unit"
}

comparison_table = pd.DataFrame([current_row, saeed_row])
comparison_table.to_csv(
    os.path.join(OUTPUT_DIR, "table_saeed_reporting_format_with_comparability.csv"),
    index=False
)
display(comparison_table)

# Internal consistency check of the published fold values.
saeed_recomputed = {
    "mean_from_5_folds": float(SAEED_AUC_FOLDS.mean()),
    "variance_sample_ddof1_from_5_folds": float(np.var(SAEED_AUC_FOLDS, ddof=1)),
    "variance_population_ddof0_from_5_folds": float(np.var(SAEED_AUC_FOLDS, ddof=0)),
    "reported_mean": SAEED_REPORTED_MEAN_AUC,
    "reported_variance": SAEED_REPORTED_VARIANCE_AUC
}
print("Saeed Table 4 consistency check:")
print(json.dumps(saeed_recomputed, indent=2))


## 8. OOF predictions + per-patient descriptive metrics

The OOF predictions remain **voxel predictions**. Per-patient ROC-AUC below summarizes voxel discrimination within each patient; it is **not** a patient-level MGMT prediction and must not be presented as equivalent to Saeed et al.'s patient-level AUC.


In [ ]:
oof = pd.concat(oof_parts, ignore_index=True)
oof.to_csv(os.path.join(OUTPUT_DIR, "development_oof_predictions.csv"), index=False)

patient_rows = []
for pid, g in oof.groupby(patient_col):
    yt = g["label"].astype(int).to_numpy()
    yp = g["prob_tumor"].to_numpy()
    patient_rows.append({
        patient_col: pid,
        "n_voxels": len(g),
        "n_non_tumor": int((yt == 0).sum()),
        "n_tumor": int((yt == 1).sum()),
        "voxel_roc_auc_within_patient": roc_auc_score(yt, yp) if np.unique(yt).size == 2 else np.nan,
        "voxel_pr_auc_within_patient": average_precision_score(yt, yp) if np.unique(yt).size == 2 else np.nan
    })

patient_metrics = pd.DataFrame(patient_rows)
patient_metrics.to_csv(os.path.join(OUTPUT_DIR, "development_oof_patient_descriptive_metrics.csv"), index=False)
display(patient_metrics.head())


## 9. Patient-wise descriptive Mean / SD / 95% CI

These confidence intervals summarize the distribution of **within-patient voxel-level discrimination metrics**. They are supplementary analyses for the current study, not a reconstruction of Saeed et al.'s patient-level MGMT evaluation.


In [ ]:
def summarize(s):
    x = pd.Series(s).dropna().astype(float).to_numpy()
    n = len(x)
    mean = x.mean() if n else np.nan
    sd = x.std(ddof=1) if n > 1 else np.nan
    se = sd / np.sqrt(n) if n > 1 else np.nan
    return dict(
        n=n,
        mean=mean,
        sd=sd,
        ci95_low=mean - 1.96 * se if n > 1 else np.nan,
        ci95_high=mean + 1.96 * se if n > 1 else np.nan,
        median=np.median(x) if n else np.nan,
        q1=np.quantile(x, 0.25) if n else np.nan,
        q3=np.quantile(x, 0.75) if n else np.nan
    )

patient_summary = pd.DataFrame([
    {"metric": "Within-patient voxel ROC-AUC", **summarize(patient_metrics["voxel_roc_auc_within_patient"])},
    {"metric": "Within-patient voxel PR-AUC", **summarize(patient_metrics["voxel_pr_auc_within_patient"])}
])
patient_summary.to_csv(
    os.path.join(OUTPUT_DIR, "development_oof_patient_descriptive_summary.csv"),
    index=False
)
display(patient_summary)


# Interpretation / next protocol

1. Use the generated **Fold 1–5 / Mean / Variance** table when describing methodological alignment with Saeed et al. (2023).
2. State explicitly that Saeed et al. perform **patient-level MGMT classification**, while this study performs **voxel-level Tumor/Non-tumor classification with patient-separated folds**.
3. **Do not report “our AUC is X% higher than Saeed”** and do not claim model superiority from these two AUCs.
4. Do not tune the model or threshold using the Locked Test set. After the CV configuration is frozen, evaluate the final model once on Locked Test.
5. For whole-tumor segmentation Dice, perform dense inference over valid voxels and reconstruct the 3D mask using `patient_id` + voxel coordinates; representative sampled voxels alone are insufficient for whole-volume Dice.
